In [ ]:
!pip install -i https://test.pypi.org/simple/ supervision==0.3.0
!pip install -q transformers
!pip install -q pytorch-lightning
!pip install -q timm

In [ ]:
import torch
from transformers import DetrForObjectDetection, DetrImageProcessor


MODEL_DIR    = "/content/drive/MyDrive/Updated DETR Model/custom-modelv2-50epochs"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DetrForObjectDetection.from_pretrained(MODEL_DIR, ignore_mismatched_sizes=True).to(device)
image_processor = DetrImageProcessor.from_pretrained(MODEL_DIR)

In [ ]:
import torch
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)

import supervision as sv
import transformers
import pytorch_lightning

print(
    "; supervision:", sv.__version__,
    "; transformers:", transformers.__version__,
    "; pytorch_lightning:", pytorch_lightning.__version__
)

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torchvision



TRAIN_DIRECTORY = "/content/drive/MyDrive/New Dataset/splits/train"
VAL_DIRECTORY = "/content/drive/MyDrive/New Dataset/splits/val"
TEST_DIRECTORY ="/content/drive/MyDrive/New Dataset/splits/test"
TRAIN_ANNOTATIONS = "/content/drive/MyDrive/New Dataset/annotations_train.json"
VAL_ANNOTATIONS = "/content/drive/MyDrive/New Dataset/annotations_val.json"
TEST_ANNOTATIONS = "/content/drive/MyDrive/New Dataset/annotations_test.json"
class CocoDetection(torchvision.datasets.CocoDetection):
    def __init__(
        self,
        image_directory_path: str,
        annotation_file_path: str,
        image_processor
    ):
        super(CocoDetection, self).__init__(image_directory_path, annotation_file_path)
        self.image_processor = image_processor

    def __getitem__(self, idx):
        images, annotations = super(CocoDetection, self).__getitem__(idx)
        image_id = self.ids[idx]
        annotations = {'image_id': image_id, 'annotations': annotations}
        encoding = self.image_processor(images=images, annotations=annotations, return_tensors="pt")
        pixel_values = encoding["pixel_values"].squeeze()
        target = encoding["labels"][0]
        return pixel_values, target
TRAIN_DATASET = CocoDetection(TRAIN_DIRECTORY, TRAIN_ANNOTATIONS, image_processor)
VAL_DATASET = CocoDetection(VAL_DIRECTORY, VAL_ANNOTATIONS, image_processor)
TEST_DATASET = CocoDetection(TEST_DIRECTORY, TEST_ANNOTATIONS, image_processor)

print("Number of training examples:", len(TRAIN_DATASET))
print("Number of validation examples:", len(VAL_DATASET))
print("Number of test examples:", len(TEST_DATASET))


In [ ]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    pixel_values = [item[0] for item in batch]
    encoding = image_processor.pad(pixel_values, return_tensors="pt")
    labels = [item[1] for item in batch]
    return {
        'pixel_values': encoding['pixel_values'],
        'pixel_mask': encoding['pixel_mask'],
        'labels': labels
    }

TRAIN_DATALOADER = DataLoader(dataset=TRAIN_DATASET, collate_fn=collate_fn, batch_size=16, shuffle=True, num_workers = 4)
VAL_DATALOADER = DataLoader(dataset=VAL_DATASET, collate_fn=collate_fn, batch_size=16,num_workers = 4)
TEST_DATALOADER = DataLoader(dataset=TEST_DATASET, collate_fn=collate_fn, batch_size=16,num_workers = 4)

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

import pytorch_lightning as pl
from transformers import DetrForObjectDetection,DetrConfig
import torch


class Detr(pl.LightningModule):

    def __init__(self, lr, lr_backbone, weight_decay):
        super().__init__()
        config = DetrConfig.from_pretrained("facebook/detr-resnet-50")
        config.bbox_loss_coefficient = 6.0
        config.giou_loss_coefficient = 2.5
        config.num_labels = len(id2label)


        self.model = DetrForObjectDetection.from_pretrained(
            pretrained_model_name_or_path=CHECKPOINT,
            config=config,
            ignore_mismatched_sizes=True
        )

        def transfer_batch_to_device(self, batch, device, dataloader_idx):
          if 'pixel_values' in batch and isinstance(batch['pixel_values'], torch.Tensor):
              if torch.isnan(batch['pixel_values']).any():
                  print(f"!!! NaN detected in pixel_values in dataloader_idx {dataloader_idx} before common_step !!!")
              if torch.isinf(batch['pixel_values']).any():
                  print(f"!!! Inf detected in pixel_values in dataloader_idx {dataloader_idx} before common_step !!!")

          # For labels:
          if 'labels' in batch and isinstance(batch['labels'], list):
              for i, label_dict in enumerate(batch['labels']):
                  if 'boxes' in label_dict and isinstance(label_dict['boxes'], torch.Tensor):
                      if torch.isnan(label_dict['boxes']).any():
                          print(f"!!! NaN detected in label boxes for sample {i} in dataloader_idx {dataloader_idx} before common_step !!!")
                      if torch.isinf(label_dict['boxes']).any():
                          print(f"!!! Inf detected in label boxes for sample {i} in dataloader_idx {dataloader_idx} before common_step !!!")
                  if 'class_labels' in label_dict and isinstance(label_dict['class_labels'], torch.Tensor):
                      # Class labels should be integers, NaN/Inf usually won't apply unless corrupted at source
                      if label_dict['class_labels'].dtype not in [torch.long, torch.int]:
                          print(f"!!! WARNING: class_labels dtype {label_dict['class_labels'].dtype} for sample {i} in dataloader_idx {dataloader_idx} is not long/int !!!")
                      # Sanity check values anyway
                      if label_dict['class_labels'].numel() > 0:
                          if label_dict['class_labels'].min() < 0 or label_dict['class_labels'].max() >= len(self.id2label):
                              print(f"!!! WARNING: class_labels for sample {i} in dataloader_idx {dataloader_idx} out of range ({label_dict['class_labels'].min().item()} to {label_dict['class_labels'].max().item()}) !!!")
          return pl.LightningModule.transfer_batch_to_device(self, batch, device, dataloader_idx)

        self.lr = lr
        self.lr_backbone = lr_backbone
        self.weight_decay = weight_decay

    def forward(self, pixel_values, pixel_mask):
        return self.model(pixel_values=pixel_values, pixel_mask=pixel_mask)

    def common_step(self, batch, batch_idx):
        pixel_values = batch["pixel_values"]
        pixel_mask = batch["pixel_mask"]
        labels = [{k: v.to(self.device) for k, v in t.items()} for t in batch["labels"]]


        outputs = self.model(pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels)

        loss_dict = outputs.loss_dict
        loss = outputs.loss

        return loss, loss_dict

    def training_step(self, batch, batch_idx):
        loss, loss_dict = self.common_step(batch, batch_idx)
        self.log("training_loss", loss)
        for k, v in loss_dict.items():
            self.log("train_" + k, v.item())

        return loss

    def validation_step(self, batch, batch_idx):
        loss, loss_dict = self.common_step(batch, batch_idx)
        self.log("validation_loss", loss)
        for k, v in loss_dict.items():
            self.log("validation_" + k, v.item())

        return loss

    def configure_optimizers(self):
        param_dicts = [
            {
                "params": [p for n, p in self.named_parameters() if "backbone" not in n and p.requires_grad]},
            {
                "params": [p for n, p in self.named_parameters() if "backbone" in n and p.requires_grad],
                "lr": self.lr_backbone,
            },
        ]
        return torch.optim.AdamW(param_dicts, lr=self.lr, weight_decay=self.weight_decay)

    def train_dataloader(self):
        return TRAIN_DATALOADER

    def val_dataloader(self):
        return VAL_DATALOADER

In [ ]:
import os

HOME = "/content/drive/MyDrive/Updated DETR Model"
CHECKPOINT_DIR = os.path.join(HOME, "checkpoints")
TB_LOG_DIR = os.path.join(HOME, "tensorboard_logs")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(TB_LOG_DIR, exist_ok=True)


In [ ]:
model = Detr(lr=1e-4, lr_backbone=1e-5, weight_decay=1e-4)

batch = next(iter(TRAIN_DATALOADER))
outputs = model(pixel_values=batch['pixel_values'], pixel_mask=batch['pixel_mask'])

In [ ]:
from pytorch_lightning.loggers import TensorBoardLogger

logger = TensorBoardLogger(
    save_dir=TB_LOG_DIR,
    name="",
    default_hp_metric=False
)
from pytorch_lightning.callbacks import ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    dirpath=CHECKPOINT_DIR,
    filename="model-epoch{epoch:02d}",
    save_top_k=-1,
    save_weights_only=False,
    every_n_epochs=1
)


In [ ]:
from pytorch_lightning import Trainer

MAX_EPOCHS = 50


trainer = Trainer(
    logger=logger,
    devices=1,
    accelerator="gpu",
    max_epochs=MAX_EPOCHS,
    gradient_clip_val=0.1,
    accumulate_grad_batches=2,
    precision=16,
    log_every_n_steps=5,
    callbacks=[checkpoint_callback]
)

In [ ]:
torch.autograd.set_detect_anomaly(True)
resume_ckpt ="/content/drive/MyDrive/Updated DETR Model/checkpoints/model-epochepoch=29.ckpt"
if resume_ckpt is not None:
    print("Resuming from checkpoint")
    trainer.fit(model, ckpt_path=resume_ckpt)
else:
    trainer.fit(model)

In [ ]:
HOME = "/content/drive/MyDrive/Updated DETR Model"
MODEL_PATH = os.path.join(HOME, 'custom-modelv2-50epochs')

In [ ]:
model.model.save_pretrained(MODEL_PATH)
image_processor.save_pretrained(MODEL_PATH)

In [ ]:
#DETR MODEL EVALUATION
!pip install -q coco_eval
from pycocotools.coco import COCO
from coco_eval import CocoEvaluator
from tqdm.notebook import tqdm
import numpy as np
import torch
import json
def convert_to_xywh(boxes):
    xmin, ymin, xmax, ymax = boxes.unbind(1)
    return torch.stack((xmin, ymin, xmax - xmin, ymax - ymin), dim=1)

def prepare_for_coco_detection(predictions):
    coco_results = []
    for original_id, prediction in predictions.items():
        if len(prediction) == 0:
            continue

        boxes = prediction["boxes"]
        boxes = convert_to_xywh(boxes).tolist()
        scores = prediction["scores"].tolist()
        labels = prediction["labels"].tolist()

        coco_results.extend(
            [
                {
                    "image_id": original_id,
                    "category_id": labels[k],
                    "bbox": box,
                    "score": scores[k],
                }
                for k, box in enumerate(boxes)
            ]
        )
    return coco_results

TEST_DATALOADER = DataLoader(dataset=TEST_DATASET, collate_fn=collate_fn, batch_size=8,num_workers = 0)
EXCLUDE_CLASS_ID = 0
model.to(DEVICE)
TEST_ANNOTATIONS_FILE_PATH = TEST_ANNOTATIONS
model.eval()
minimal_info = {
    "description": "Custom Vehicle Detection Dataset",
    "url": "http://example.com/your_dataset",
    "version": "1.0",
    "year": 2025,
    "contributor": "Your Name/Organization",
    "date_created": "2025/07/04"
    }

coco_data_to_save = None

if isinstance(TEST_DATALOADER.dataset.coco, dict):
    if 'info' not in TEST_DATALOADER.dataset.coco:
        TEST_DATALOADER.dataset.coco['info'] = minimal_info
    if 'licenses' not in TEST_DATALOADER.dataset.coco:
        TEST_DATALOADER.dataset.coco['licenses'] = [
            {"id": 1, "name": "Unknown License", "url": "N/A"}
        ]
    coco_data_to_save = TEST_DATALOADER.dataset.coco

elif isinstance(TEST_DATALOADER.dataset.coco, COCO):
    if 'info' not in TEST_DATALOADER.dataset.coco.dataset:
        TEST_DATALOADER.dataset.coco.dataset['info'] = minimal_info
    if 'licenses' not in TEST_DATALOADER.dataset.coco.dataset:
        TEST_DATALOADER.dataset.coco.dataset['licenses'] = [
            {"id": 1, "name": "Unknown License", "url": "N/A"}
        ]
    coco_data_to_save = TEST_DATALOADER.dataset.coco.dataset

else:
    print("WARNING: TEST_DATALOADER.dataset.coco is neither a dict nor a pycocotools.COCO object. Cannot inject 'info'.")

if coco_data_to_save is not None:
    output_dir = os.path.dirname(TEST_ANNOTATIONS_FILE_PATH)
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    with open(TEST_ANNOTATIONS_FILE_PATH, 'w') as f:
        json.dump(coco_data_to_save, f, indent=4)
    print(f"COCO annotations updated and saved to: {TEST_ANNOTATIONS_FILE_PATH}")
else:
    print("No COCO data was identified or modified to be saved.")


print("--- Inspecting TEST_DATALOADER.dataset.coco AFTER adding info ---")
print(f"Type of TEST_DATALOADER.dataset.coco: {type(TEST_DATALOADER.dataset.coco)}")

if hasattr(TEST_DATALOADER.dataset.coco, 'dataset'):
    print(f"Keys in TEST_DATALOADER.dataset.coco.dataset: {TEST_DATALOADER.dataset.coco.dataset.keys()}")
    if 'annotations' not in TEST_DATALOADER.dataset.coco.dataset:
        print("!!! WARNING: 'annotations' key is missing in TEST_DATALOADER.dataset.coco.dataset !!!")
    if 'images' not in TEST_DATALOADER.dataset.coco.dataset:
        print("!!! WARNING: 'images' key is missing in TEST_DATALOADER.dataset.coco.dataset !!!")
    if 'categories' not in TEST_DATALOADER.dataset.coco.dataset:
        print("!!! WARNING: 'categories' key is missing in TEST_DATALOADER.dataset.coco.dataset !!!")
else:
    if isinstance(TEST_DATALOADER.dataset.coco, dict):
        print(f"Keys in TEST_DATALOADER.dataset.coco (it's a dict): {TEST_DATALOADER.dataset.coco.keys()}")
        if 'annotations' not in TEST_DATALOADER.dataset.coco:
            print("!!! WARNING: 'annotations' key is missing in TEST_DATALOADER.dataset.coco !!!")
        if 'images' not in TEST_DATALOADER.dataset.coco:
            print("!!! WARNING: 'images' key is missing in TEST_DATALOADER.dataset.coco !!!")
        if 'categories' not in TEST_DATALOADER.dataset.coco:
            print("!!! WARNING: 'categories' key is missing in TEST_DATALOADER.dataset.coco !!!")
    else:
        print("!!! UNEXPECTED: TEST_DATALOADER.dataset.coco does not have a 'dataset' attribute and is not a dict. !!!")


evaluator = CocoEvaluator(coco_gt=TEST_DATASET.coco, iou_types=["bbox"])

NMS_IOU_THRESHOLD = 0.2
CONFIDENCE_TRESHOLD_EVAL = 0.85

for idx, batch in enumerate(tqdm(TEST_DATALOADER)):
    pixel_values = batch["pixel_values"].to(DEVICE)
    pixel_mask = batch["pixel_mask"].to(DEVICE)
    labels = [{k: v.to(DEVICE) for k, v in t.items()} for t in batch["labels"]]

    with torch.no_grad():
      outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask)

    orig_target_sizes = torch.stack([target["orig_size"] for target in labels], dim=0)
    results = image_processor.post_process_object_detection(
        outputs,
        target_sizes=orig_target_sizes,
        threshold=CONFIDENCE_TRESHOLD_EVAL
    )
    image_predictions_for_eval = {}

    for target, output in zip(labels, results):
        image_id = target['image_id'].item()

        detections = sv.Detections(
            xyxy=output['boxes'].cpu().numpy(),
            confidence=output['scores'].cpu().numpy(),
            class_id=output['labels'].cpu().numpy()
        )

        detections = detections.with_nms(
            threshold=NMS_IOU_THRESHOLD
        )
        detections = detections[detections.class_id != EXCLUDE_CLASS_ID]

        processed_output = {
            "boxes": torch.from_numpy(detections.xyxy).to(DEVICE),
            "scores": torch.from_numpy(detections.confidence).to(DEVICE),
            "labels": torch.from_numpy(detections.class_id).to(DEVICE)
        }

        image_predictions_for_eval[image_id] = processed_output

    coco_results_for_eval = prepare_for_coco_detection(image_predictions_for_eval)
    evaluator.update(coco_results_for_eval)
evaluator.synchronize_between_processes()
evaluator.accumulate()
evaluator.summarize()